# TokenAI — BERTurk L1 Sınıflandırıcı Eğitimi

**Çalıştırmadan önce:** Üst menü → **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** → Kaydet.
Sonra **Çalışma zamanı → Tümünü çalıştır**. Drive izni isterse onaylayın.

Beklenen süre: 15–25 dakika. Bittiğinde Drive'daki `ml/outputs/` klasöründe şunlar olur:
- `model.zip` → bilgisayarınızda `LUAS/ml/models/` içine çıkarın
- `report.md` → `LUAS/ml/reports/` içine koyun ve Claude'a haber verin


In [ ]:
# 1) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

import os
ML_DIR = '/content/drive/MyDrive/ml'   # ml klasörünü Drive'da başka yere koyduysanız burayı değiştirin
assert os.path.exists(f'{ML_DIR}/scripts/train.py'), f'{ML_DIR} bulunamadı — ml klasörünü Drive ana dizinine yükleyin'
print('ml klasörü bulundu:', ML_DIR)

In [ ]:
# 2) GPU kontrolü
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — Çalışma zamanı > T4 GPU seçin!')

In [ ]:
# 3) Dosyaları hızlı yerel diske kopyala ve gerekli paketleri kur
!rm -rf /content/ml && cp -r "$ML_DIR" /content/ml
!pip install -q -r /content/ml/requirements-train.txt

In [ ]:
# 4) Veriyi hazırla (tekrar temizliği, grup bazlı bölme, test sızıntı kontrolü)
!cd /content/ml && python scripts/prepare_data.py

In [ ]:
# 5) Eğit (BERTurk ince ayar)
!cd /content/ml && python scripts/train.py --epochs 4 --lr 3e-5 --batch_size 16

In [ ]:
# 6) Kalibre et, eşik seç, insan yazımı test setinde değerlendir
!cd /content/ml && python scripts/evaluate.py --target_precision 0.98

In [ ]:
# 7) Sonuçları Drive'a kaydet
!cd /content/ml/outputs && rm -f model.zip && zip -qr model.zip model
!mkdir -p "$ML_DIR/outputs" && cp /content/ml/outputs/model.zip /content/ml/outputs/report.md /content/ml/data/splits/stats.json "$ML_DIR/outputs/"
!ls -lh "$ML_DIR/outputs"
print('Bitti. Drive > ml > outputs klasöründen model.zip ve report.md dosyalarını indirin.')

In [ ]:
# 8) (İsteğe bağlı) Modeli elle dene
import json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
md_ = '/content/ml/outputs/model'
cal = json.load(open(f'{md_}/calibration.json'))
tok = AutoTokenizer.from_pretrained(md_); mdl = AutoModelForSequenceClassification.from_pretrained(md_).eval()
def sor(t):
    with torch.no_grad():
        p = torch.softmax(mdl(**tok(t, return_tensors='pt')).logits / cal['temperature'], -1)[0]
    lab = cal['labels'][int(p.argmax())]
    cache = lab == 'genel' and float(p[0]) >= cal['tau']
    print(f"{t!r:55} -> {lab:9} p(genel)={float(p[0]):.3f}  önbellek={'EVET' if cache else 'hayır'}")
for t in ['Türkiye\'nin başkenti neresidir?', 'Benim adım ne?', 'Güncel altın fiyatı nedir?', 'Bana bir şiir yaz', 'fotosentez nasıl olur']:
    sor(t)